# Notebook 01 - Static Characterization

**Milestone:** HV-004  
**Technical Gate:** SC-001  
**Primary Engineering Question:** EQ-02 - Static Measurement Performance  
**Secondary Engineering Question:** EQ-04 - Hysteresis / Direction Dependence

This notebook characterizes the pre-calibration static measurement behavior of the DUT using only qualified analyst-visible evidence from the frozen `VSM-LAB-DATA-v1.1` release.

The notebook must preserve the analyst-blind boundary and the calibration/validation separation established during DQ-001.

## 1. Engineering Question

### Primary Question

How large is the DUT measurement error, and how does it vary with relative humidity and temperature?

### Secondary Question

Does the static error differ between increasing and decreasing humidity directions under matched conditions?

The primary observed error is defined as:

```text
Error_RH = RH_DUT - RH_Reference
```

A positive error means that the DUT indication is higher than the reference indication.

The reference indication is not treated as exact physical truth.

## 2. Why This Matters

Static characterization establishes the baseline measurement behavior of the DUT before any calibration correction is developed.

This stage is intended to determine whether the observed DUT-reference difference is approximately constant or whether it varies systematically with operating conditions such as relative humidity, temperature, and direction of environmental change.

A defensible characterization must also distinguish sample-level variation from independent experimental replication and must prevent calibration or validation data from leaking into the characterization analysis.

## 3. Scope and Analysis Boundary

### Included in HV-004

- qualified `STATIC` observations
- `CHARACTERIZATION` observations only
- pre-calibration DUT-reference error
- RH dependence
- temperature dependence
- matched Up/Down direction dependence
- day/run context
- descriptive static error metrics

### Deferred to later milestones

- formal repeatability/reproducibility variance decomposition
- calibration-model fitting
- model freeze
- independent validation
- dynamic-response analysis
- stability/drift analysis
- final uncertainty budget
- final DUT compliance decision

The working analysis frame must contain zero `CALIBRATION` and zero `VALIDATION` rows.

## 4. Input Data

Primary measurement file:

```text
data/raw/vsm_lab_data_v1_1/static_measurements.csv
```

Supporting experimental-design file:

```text
data/raw/vsm_lab_data_v1_1/test_matrix.csv
```

The frozen release passed DQ-001 before this notebook was started.

### SC-01 Objective

Load the frozen static data, isolate the qualified characterization subset, and verify that the working analysis frame respects the HV-004 role boundary.

In [ ]:
from pathlib import Path

import pandas as pd


# ------------------------------------------------------------
# Locate project root and frozen dataset release
# ------------------------------------------------------------

current_dir = Path.cwd()

candidate_roots = [
    current_dir,
    current_dir.parent,
]

project_root = next(
    (
        path for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

raw_data_dir = (
    project_root
    / "data"
    / "raw"
    / "vsm_lab_data_v1_1"
)

static_path = raw_data_dir / "static_measurements.csv"
test_matrix_path = raw_data_dir / "test_matrix.csv"

print(f"Project root : {project_root}")
print(f"Dataset path : {raw_data_dir}")
print(f"Static file  : {static_path.name}")
print(f"Test matrix  : {test_matrix_path.name}")

### 4.1 Load Static Measurements and Test Matrix

The raw static file contains multiple analysis roles. HV-004 does not use all of them.

The purpose of this step is to load the source data without modifying the frozen files and then establish the role composition before filtering.

In [ ]:
# ------------------------------------------------------------
# Load frozen analyst-facing inputs
# ------------------------------------------------------------

static_df = pd.read_csv(static_path)
test_matrix_df = pd.read_csv(test_matrix_path)

print(f"Static rows       : {len(static_df):,}")
print(f"Static columns    : {static_df.shape[1]}")
print(f"Static Test_IDs   : {static_df['Test_ID'].nunique():,}")
print(f"Test-matrix rows  : {len(test_matrix_df):,}")

print("\nStatic Analysis_Role counts:")
display(
    static_df["Analysis_Role"]
    .value_counts(dropna=False)
    .rename_axis("Analysis_Role")
    .reset_index(name="Rows")
)

print("\nStatic Test_ID counts by Analysis_Role:")
display(
    static_df.groupby("Analysis_Role")["Test_ID"]
    .nunique()
    .rename("Test_IDs")
    .reset_index()
)

### 4.2 Isolate the Characterization Working Frame

Only `STATIC / CHARACTERIZATION` observations are released into the HV-004 performance-analysis frame.

This is a deliberate methodological boundary:

- `CHARACTERIZATION` describes the raw DUT behavior.
- `CALIBRATION` is reserved for later model development.
- `VALIDATION` remains independent until the calibration model has been frozen.

This separation prevents data leakage.

In [ ]:
# ------------------------------------------------------------
# Isolate qualified STATIC / CHARACTERIZATION observations
# ------------------------------------------------------------

characterization_df = static_df.loc[
    (static_df["Test_Family"] == "STATIC")
    & (static_df["Analysis_Role"] == "CHARACTERIZATION")
].copy()

role_counts_in_working_frame = (
    characterization_df["Analysis_Role"]
    .value_counts(dropna=False)
)

test_family_counts_in_working_frame = (
    characterization_df["Test_Family"]
    .value_counts(dropna=False)
)

print(f"Characterization rows     : {len(characterization_df):,}")
print(f"Characterization Test_IDs : {characterization_df['Test_ID'].nunique():,}")

print("\nTest_Family values in working frame:")
display(
    test_family_counts_in_working_frame
    .rename_axis("Test_Family")
    .reset_index(name="Rows")
)

print("\nAnalysis_Role values in working frame:")
display(
    role_counts_in_working_frame
    .rename_axis("Analysis_Role")
    .reset_index(name="Rows")
)

### 4.3 SC-01 Role-Isolation Check

This check verifies completion criterion **C01 - Qualified characterization data only** and supports **C09 - Calibration and validation leakage is prevented**.

The working frame passes this check only if:

1. at least one characterization row is present;
2. every row has `Test_Family == STATIC`;
3. every row has `Analysis_Role == CHARACTERIZATION`;
4. no calibration rows are present;
5. no validation rows are present.

In [ ]:
# ------------------------------------------------------------
# SC-01 role-isolation gate
# ------------------------------------------------------------

has_rows = len(characterization_df) > 0

static_only = (
    characterization_df["Test_Family"]
    .eq("STATIC")
    .all()
)

characterization_only = (
    characterization_df["Analysis_Role"]
    .eq("CHARACTERIZATION")
    .all()
)

calibration_rows_in_working_frame = (
    characterization_df["Analysis_Role"]
    .eq("CALIBRATION")
    .sum()
)

validation_rows_in_working_frame = (
    characterization_df["Analysis_Role"]
    .eq("VALIDATION")
    .sum()
)

sc01_role_isolation_pass = (
    has_rows
    and static_only
    and characterization_only
    and calibration_rows_in_working_frame == 0
    and validation_rows_in_working_frame == 0
)

print(f"Working frame has rows       : {has_rows}")
print(f"STATIC only                  : {static_only}")
print(f"CHARACTERIZATION only        : {characterization_only}")
print(
    "CALIBRATION rows in frame    : "
    f"{calibration_rows_in_working_frame}"
)
print(
    "VALIDATION rows in frame     : "
    f"{validation_rows_in_working_frame}"
)

print(
    "\nSC-01 role-isolation check    : "
    f"{'PASS' if sc01_role_isolation_pass else 'FAIL'}"
)

if not sc01_role_isolation_pass:
    raise RuntimeError(
        "SC-01 role-isolation check failed. "
        "Do not continue static characterization."
    )

### 4.4 Experimental-Condition Inventory

Before calculating performance metrics, inspect the experimental factors represented in the characterization subset.

For this notebook, an experimental condition is defined by a combination of factor levels such as RH setpoint, temperature setpoint, direction, day, and run.

This inventory is descriptive. It does not yet estimate sensor performance.

In [ ]:
# ------------------------------------------------------------
# Inspect characterization design factors
# ------------------------------------------------------------

design_factors = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
    "Day_ID",
    "Run_ID",
    "Chamber_State",
]

design_inventory = []

for column in design_factors:
    values = (
        characterization_df[column]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )
    
    design_inventory.append(
        {
            "Factor": column,
            "Number_of_Levels": len(values),
            "Levels": values,
        }
    )

design_inventory_df = pd.DataFrame(design_inventory)

display(design_inventory_df)

### SC-01 Engineering Interpretation

Complete this interpretation only after running the cells above.

Questions to answer:

- Did the expected characterization evidence load successfully?
- How many rows and Test_IDs are available?
- Are calibration and validation observations absent from the working frame?
- Which RH, temperature, direction, day, and run levels define the characterization design?
- Is there any reason to stop before creating the measurement-error variable?

**Do not interpret DUT performance yet.** SC-01 establishes the analysis boundary; sensor-performance interpretation begins after the error variable and Test_ID-level summaries are created.

## 5. Measurement Model and Error Convention

**Next implementation stage: SC-02**

The next step will create and verify:

```text
Error_RH = RH_DUT - RH_Reference
Abs_Error_RH = abs(Error_RH)
```

No calibration correction will be introduced in HV-004.

## 6. Unit of Analysis

**Next implementation stage: SC-03**

Sample-level observations and Test_ID-level experimental summaries will be kept conceptually distinct to avoid pseudoreplication.

## 7. Static Data Preparation

To be implemented after SC-01 is verified.

## 8. Overall Error Characterization

To be implemented after the error variables and Test_ID-level summary table are verified.

## 9. RH Dependence

To be implemented after the overall baseline characterization.

## 10. Temperature Dependence

To be implemented after RH dependence is characterized.

## 11. Direction Dependence

To be implemented using matched Up/Down conditions.

## 12. Day / Run Context

To be implemented as contextual inspection only. Formal R&R analysis remains in Notebook 02.

## 13. Results Summary

To be completed after all HV-004 analyses are implemented and verified.

## 14. Engineering Interpretation

To be completed after the numerical and graphical results are available.

## 15. What I Learned

To be completed during milestone closeout.

## 16. Key Metrology Concepts

To be completed during milestone closeout.

## 17. Questions I Should Be Able to Answer

To be completed during milestone closeout.

## 18. Connection to the Uncertainty Budget

To be completed during milestone closeout.

## 19. Final Engineering Takeaway

To be completed during milestone closeout.

## 20. Technical Discussion / Interview Notes

To be completed during milestone closeout.

## 21. SC-001 Gate

The final SC-001 decision will be made only after all twelve completion criteria are evaluated and the notebook passes Restart Kernel -> Run All.